# SEAL vector construction

Generate reasoning traces for the paper's 1,000 MATH training problems, classify paragraph breaks as execution, reflection or transition, and average their captured hidden states. This notebook regenerates the three GGUF vectors used by `evaluate.ipynb`.

Use the same DeepSeek-R1-Distill-Qwen-1.5B model and input dataset as the original experiment. This is the expensive reconstruction step; evaluation can reuse the committed vectors. The refactored construction notebook has not been rerun at paper scale.


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
from common import make_prompts
from vllm import LLM, SamplingParams
from vllm.capture import SelectSpec

from easysteer.capture import capture_batches, release_capture_cache
from easysteer.extraction import StatisticalControlVector

MODEL = os.environ.get("EASYSTEER_MODEL", "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
DATA_DIR = Path(os.environ.get("EASYSTEER_DATA_DIR", "."))
MAX_TOKENS = 8192
CAPTURE_BATCH_SIZE = 32

llm = LLM(
    model=MODEL, tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1"))
)
tokenizer = llm.get_tokenizer()
problems = json.loads((DATA_DIR / "math_train_1000.json").read_text())
prompts = make_prompts(tokenizer, problems)
outputs = llm.generate(
    prompts,
    SamplingParams(temperature=0, max_tokens=MAX_TOKENS, skip_special_tokens=False),
    steering=False,
)
# Capture the actual generated sequence, without a text-to-token round trip.
trace_ids = [
    prompt["prompt_token_ids"] + output.outputs[0].token_ids
    for prompt, output in zip(prompts, outputs, strict=True)
]
del outputs

## Classify paragraph breaks

The existing keyword rules assign each paragraph to the break token preceding it. Token IDs and absolute positions are reused unchanged for capture.


In [ ]:
TRANSITION_KEYWORDS = [
    "alternatively",
    "think differently",
    "another way",
    "another approach",
    "another method",
    "another solution",
    "another strategy",
    "another technique",
]
REFLECTION_KEYWORDS = [
    "wait",
    "verify",
    "make sure",
    "hold on",
    "think again",
    "'s correct",
    "'s incorrect",
    "let me check",
    "seems right",
]


def classify(segment):
    lower = segment.lower()
    if any(k in lower for k in TRANSITION_KEYWORDS):
        return "Transition"
    if any(k in lower for k in REFLECTION_KEYWORDS):
        return "Reflection"
    return "Execution"


category_by_position = []
for ids in trace_ids:
    tokens = tokenizer.convert_ids_to_tokens(ids)
    positions = [i for i, t in enumerate(tokens) if t.endswith("ĊĊ")]
    by_pos = {}
    for j, pos in enumerate(positions):
        end = positions[j + 1] if j + 1 < len(positions) else len(ids)
        segment = tokenizer.decode(ids[pos + 1 : end], skip_special_tokens=True)
        by_pos[pos] = classify(segment.strip())
    category_by_position.append(by_pos)

total = sum(len(b) for b in category_by_position)
print(f"{total} paragraph breaks across {len(trace_ids)} traces")

## Capture and average

Capture only classified paragraph breaks, in batches bounded by prompt count and the default 256 MiB raw-data budget. Per-category sums are retained, and each batch is released before the next one. Use each result's `sample_indices` to map byte-sized batches back to their traces. Every paragraph break contributes equally; pooling each trace to one row would change the experiment.


In [ ]:
sums = {"Transition": {}, "Reflection": {}, "Execution": {}}
counts = {category: 0 for category in sums}
active_indices = [i for i, mapping in enumerate(category_by_position) if mapping]
batches = capture_batches(
    llm,
    ({"prompt_token_ids": trace_ids[i]} for i in active_indices),
    batch_size=CAPTURE_BATCH_SIZE,
    per_prompt_selects=(
        SelectSpec(prompt_positions=list(category_by_position[i]))
        for i in active_indices
    ),
    max_tokens=1,
    steering=False,
)
for result in batches:
    for sample_index, source_index in enumerate(result.sample_indices):
        mapping = category_by_position[active_indices[source_index]]
        for row_index, position in enumerate(result.sample_positions(sample_index)):
            category = mapping[position]
            counts[category] += 1
            for layer in result.layer_ids:
                row = result.token(sample_index, layer, row_index).float().numpy()
                if layer not in sums[category]:
                    sums[category][layer] = np.zeros_like(row)
                sums[category][layer] += row
                del row
    print(f"Processed {result.sample_indices[-1] + 1}/{len(active_indices)} traces")
    del result
release_capture_cache(llm)

for category, per_layer in sums.items():
    if not counts[category]:
        raise ValueError(f"No paragraph breaks classified as {category}")
    vector = StatisticalControlVector(
        method="Average",
        directions={
            layer: total / counts[category] for layer, total in per_layer.items()
        },
        metadata={"num_vectors_averaged": counts[category]},
        model_type=MODEL,
        component="hidden_states",
    )
    vector.export_gguf(f"{category.lower()}_avg_vector.gguf")
    print(f"{category}: averaged {counts[category]} rows")
